# 09 · Systemic stale-ffill audit (H5) — M3, M5

**Purpose.** В M2 мы нашли паттерн: непрерывная фича строится из нерегулярных событий и forward-fillّ ится → значение «протухает», но PCA видит персистентность stale-значения как дисперсию. Проверяем, есть ли тот же эффект в M3 и M5.

**What to look for.**
- `unchanged_frac` (доля дней без изменения = forward-fill) и `median_age`
- `Sp_all` vs `Sp_fresh` (на свежих днях age≤3): коллапс = stale-артефакт; рост = сигнал размывается zero-fill
- какие фичи в whitelist (видит PCA), какие — кандидаты

> Метрика та же, что в 08: fresh/all < 1 → артефакт; fresh/all > 1 → сигнал ослаблен импутацией.

In [1]:
import sys, os
from pathlib import Path
_here=Path.cwd(); _root=next((p for p in [_here,*_here.parents] if (p/'data'/'processed').is_dir()),_here)
os.chdir(_root); sys.path.insert(0,str(_root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.width',180); pd.set_option('display.max_columns',80); pd.set_option('display.max_rows',80)
import importlib
from lab import utils as u
importlib.reload(u)
print('root:',_root.name)

root: MathMode_LiquidityCatchers_RuLiquiditySentinel


In [2]:
d=u.load_final_dataset(); base_wl=u.available_whitelist(d)
art_A=u.fit_lsi_like_model(d,base_wl); lsi=art_A['lsi']
def audit(series, age=None, name=''):
    s=pd.to_numeric(series,errors='coerce'); unchg=float((s.diff()==0).mean()); sp_all=u.spearman(s.values,lsi)
    if age is not None:
        m=(pd.to_numeric(age,errors='coerce')<=3).values & (s.values!=0)
        sp_fr=u.spearman(s.values[m],lsi[m]) if m.sum()>30 else float('nan'); nfr=int(m.sum())
    else: sp_fr,nfr=float('nan'),-1
    return {'feature':name,'whitelist':name in base_wl,'unchanged':round(unchg,3),'Sp_all':round(sp_all,3),
            'Sp_fresh':round(sp_fr,3),'fresh/all':round(abs(sp_fr)/abs(sp_all),2) if (sp_all and not np.isnan(sp_fr) and abs(sp_all)>1e-6) else None}
print('baseline ready')

baseline ready


## M5 — Roskazna / CBR liquidity
Roskazna — депозитные аукционы Казначейства (post-2021). `days_since_last_roskazna_auction` — возраст.

In [3]:
m5=pd.read_csv('data/processed/m5_features.csv'); m5['date']=pd.to_datetime(m5['date'],dayfirst=True,format='mixed')
dd=d.merge(m5[['date','days_since_last_roskazna_auction']],on='date',how='left'); age_rk=dd['days_since_last_roskazna_auction']
print('median days_since_roskazna:',float(age_rk.median()),'| <=3d frac:',round(float((age_rk<=3).mean()),3),'(roskazna почти ежедневный post-2021)')
m5_cols=[c for c in d.columns if c.startswith('m5_')]
rows=[audit(d[c], age=age_rk if 'roskazna' in c.lower() else None, name=c) for c in m5_cols]
pd.DataFrame(rows)

median days_since_roskazna: 0.0 | <=3d frac: 0.392 (roskazna почти ежедневный post-2021)


/Users/nikitabaslykov/Documents/Работа/Казначейство/MathMode_LiquidityCatchers_RuLiquiditySentinel/lab/utils.py:217: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(stats.spearmanr(a[m], b[m]).correlation)


,feature,whitelist,unchanged,Sp_all,Sp_fresh,fresh/all
0,m5_liquidity_deficit_surplus_bln_rub_lag_1d,False,0.002,0.008,NaN,NaN
1,m5_liquidity_deficit_surplus_bln_rub_change_1d,False,0.000,-0.011,NaN,NaN
2,m5_liquidity_deficit_surplus_bln_rub_change_5d,False,0.000,-0.031,NaN,NaN
3,m5_cbr_liquidity_stress_mad_score,True,0.041,-0.342,NaN,NaN
4,m5_cbr_liquidity_drain_mad_score,True,0.010,0.036,NaN,NaN
5,m5_budget_funds_total_mln_rub_lagged,False,0.952,0.502,NaN,NaN
6,m5_budget_funds_total_mln_rub_change_lagged,False,0.952,-0.068,NaN,NaN
7,m5_budget_funds_total_mln_rub_pct_change_lagged,False,0.952,-0.057,NaN,NaN
8,m5_budget_funds_rub_mln_rub_lagged,False,0.952,0.194,NaN,NaN
9,m5_budget_funds_rub_mln_rub_change_lagged,False,0.952,0.019,NaN,NaN


### Roskazna rolling-фичи (НЕ в whitelist) — кандидаты с pre-2021 zero-split

In [4]:
m5d=d.merge(m5,on='date',how='left',suffixes=('','_raw'))
cand=['roskazna_first_leg_rolling_30d_mln_rub','roskazna_net_flow_rolling_30d_mln_rub',
      'roskazna_cover_ratio_lag_1d','roskazna_bidders_count_lag_1d','budget_funds_total_mln_rub_lagged']
pd.DataFrame([audit(m5d[c],age=age_rk,name=c) for c in cand if c in m5d.columns])

,feature,whitelist,unchanged,Sp_all,Sp_fresh,fresh/all
0,roskazna_first_leg_rolling_30d_mln_rub,False,0.612,0.664,-0.033,0.05
1,roskazna_net_flow_rolling_30d_mln_rub,False,0.618,0.074,-0.094,1.27
2,roskazna_cover_ratio_lag_1d,False,0.001,-0.148,-0.148,1.00
3,roskazna_bidders_count_lag_1d,False,0.024,-0.187,-0.187,1.00
4,budget_funds_total_mln_rub_lagged,False,0.952,0.502,0.150,0.30


## M3 — OFZ аукционы (event-driven)

In [5]:
af=d['m3_auction_flag'].fillna(0).values if 'm3_auction_flag' in d.columns else np.zeros(len(d))
age_m3=np.empty(len(d)); last=-10**9
for i,v in enumerate(af):
    if v==1: last=i
    age_m3[i]=i-last if last>-10**8 else 9999
age_m3=pd.Series(age_m3)
print('median age since m3 auction:',float(age_m3.median()),'| <=3d frac:',round(float((age_m3<=3).mean()),3))
m3_cols=[c for c in d.columns if c.startswith('m3_')]
pd.DataFrame([audit(d[c], age=age_m3 if c!='m3_auction_flag' else None, name=c) for c in m3_cols])

median age since m3 auction: 3.0 | <=3d frac: 0.581


/Users/nikitabaslykov/Documents/Работа/Казначейство/MathMode_LiquidityCatchers_RuLiquiditySentinel/lab/utils.py:217: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(stats.spearmanr(a[m], b[m]).correlation)


,feature,whitelist,unchanged,Sp_all,Sp_fresh,fresh/all
0,m3_demand_amount,False,0.708,0.078,0.200,2.57
1,m3_offered_amount,False,0.708,0.082,0.330,4.04
2,m3_placed_amount,False,0.714,0.075,0.200,2.67
3,m3_weighted_yield,False,0.000,0.370,0.370,1.00
4,m3_cover_ratio,False,0.000,-0.234,-0.234,1.00
5,m3_yield_spread,False,0.000,0.030,0.030,1.00
6,m3_Flag_Nedospros,True,0.830,0.103,NaN,NaN
7,m3_Flag_Perespros,True,0.922,-0.004,NaN,NaN
8,m3_MAD_score_cover,False,0.711,0.051,0.306,5.97
9,m3_MAD_score_yield_spread,False,0.710,-0.015,-0.016,1.04


## Verdict H5
**M5:**
- `m5_cbr_liquidity_stress_mad_score` (whitelist) — **живая** (daily, unchanged 4%, Sp −0.34). Рабочая лошадка M5.
- `m5_roskazna_net_flow_stress_mad_score` (whitelist) — **stale+мёртвая**: unchanged 74%, Sp 0.03, fresh 0.03. Дохлый вес, кандидат на исключение.
- `m5_Flag_Budget_Drain` (whitelist) — sparse бинарный (monthly), Sp 0.36 — ок как флаг.
- **Roskazna rolling-фичи (НЕ в whitelist)** — классический stale-артефакт: Sp_all 0.59–0.66 → Sp_fresh ~0.05 (fresh/all 0.05–0.15). Высокая корреляция целиком из pre-2021 zero-split, не из сигнала. **Не добавлять.**
- Подозреваемые `roskazna_*_lag_1d` — на деле **НЕ stale** (roskazna почти ежедневный post-2021, median age 0); суждение H5 о них опровергнуто, но сигнал слабый (−0.15…−0.19).

**M3 — обратная проблема (не stale, а dilution):**
- `m3_cover_stress_score` (whitelist): Sp_all −0.05 → **Sp_fresh −0.31** (fresh/all ≈6!). Сигнал РЕАЛЬНЫЙ на днях аукционов, но **размыт zero-fill** на 71% не-аукционных дней. Не артефакт — недоиспользование.
- `m3_yield_stress_score` — слабый и на свежих (−0.02).

**Итог:** системный stale-ffill подтверждён в M5 (roskazna rolling — но они вне whitelist; net_flow_stress в whitelist стух). M3 — зеркальная проблема: хороший сигнал глушится zero-fill (нужен другой фикс — event-aware агрегация/доступность, а не forward-fill). M2-фикс остаётся приоритетом.